In [1]:
# 1. Cài đặt thư viện huấn luyện
!pip install -q transformers peft torch datasets librosa soundfile tqdm phonemizer sea-g2p
!pip install -q git+https://github.com/Neuphonic/NeuCodec.git
!pip install -U bitsandbytes

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.2/48.2 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 89.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.8/103.8 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.5/21.5 MB 79.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.5/69.5 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.7/216.7 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 615.4/615.4 kB 29.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>

In [2]:
from sea_g2p import Normalizer, G2P

print("⚙️ Đang khởi tạo bộ xử lý Tiếng Việt (sea-g2p)...")
norm = Normalizer()
g2p_engine = G2P()

def normalize_text(text):
    return norm.normalize(text)

def phonemize_with_dict(text):
    return g2p_engine.convert(text)

print("Text Engine đã sẵn sàng!")

⚙️ Đang khởi tạo bộ xử lý Tiếng Việt (sea-g2p)...
Text Engine đã sẵn sàng!


In [3]:
import os
import torch
import librosa
import json
import re
import pandas as pd
from neucodec import NeuCodec
from tqdm import tqdm
from huggingface_hub import login
login("hf_YOUR_HUGGINGFACE_TOKEN_HERE")
# ==========================================
# 🎯 1. CẤU HÌNH ĐẦU VÀO & NGƯỜI NÓI
# ==========================================
INPUT_DATA_DIR = "/kaggle/input/datasets/phmngcduy/data-utterances" 
TARGET_SPEAKER = "SPEAKER_00"

OUTPUT_DIR = "/kaggle/working/dataset_encoded"
os.makedirs(OUTPUT_DIR, exist_ok=True)

METADATA_PATH = os.path.join(INPUT_DATA_DIR, "metadata.csv")
ENCODED_PATH = os.path.join(OUTPUT_DIR, "metadata_encoded.csv")

# ==========================================
# 🛠️ 2. MÃ HÓA VÀ CHẨN ĐOÁN LỖI TẬN GỐC
# ==========================================
print(f"🔍 Bắt đầu KIỂM TRA LỖI từ: {INPUT_DATA_DIR}")

df_full = pd.read_csv(METADATA_PATH)
df_target = df_full[df_full['speaker_id'] == TARGET_SPEAKER]

print(f"✅ Đã tìm thấy {len(df_target)} câu nói của {TARGET_SPEAKER} trong file CSV. Bắt đầu mã hóa...")

device = "cuda" if torch.cuda.is_available() else "cpu"
codec = NeuCodec.from_pretrained("neuphonic/neucodec").to(device)
codec.eval()

valid_samples = []

# --- BỘ ĐẾM LỖI ---
err_path = 0
err_dur = 0
err_text = 0
err_librosa = 0
err_encode = 0

for index, row in tqdm(df_target.iterrows(), total=len(df_target), desc="Đang mã hóa"):
    filename = str(row['audio_file'])
    duration_sec = float(row['duration_sec'])
    text = str(row['text']).strip()
    
    audio_path = os.path.join(INPUT_DATA_DIR, filename)
    
    # 1. Kiểm tra file có tồn tại thật trong ổ cứng không?
    if not os.path.exists(audio_path):
        err_path += 1
        continue
        
    # 2. Mở rộng bộ lọc thời lượng: Nhận file từ 1 giây đến 20 giây
    if not (1.0 <= duration_sec <= 20.0):
        err_dur += 1
        continue
        
    # 3. Kiểm tra văn bản rỗng
    if not text:
        err_text += 1
        continue
        
    # Tự động thêm dấu chấm để AI dễ đọc
    if text[-1] not in ".,?!":
        text += "."
        
    # 4. Thử đọc file âm thanh
    try:
        wav, _ = librosa.load(audio_path, sr=16000, mono=True)
    except Exception as e:
        if err_librosa == 0: print(f"\n❌ Phát hiện file audio hỏng ({filename}): {e}")
        err_librosa += 1
        continue

    # 5. Thử mã hóa bằng NeuCodec
    try:
        # SỬA LỖI CHÍ MẠNG: Ép wav_tensor vào 'cuda' cùng hệ với codec
        wav_tensor = torch.from_numpy(wav).float().unsqueeze(0).unsqueeze(0).to(device)
        
        with torch.no_grad():
            codes = codec.encode_code(wav_tensor)
            codes = codes.squeeze(0).squeeze(0).cpu().numpy().flatten().tolist()
            codes = [int(x) for x in codes]
        
        if codes and all(0 <= c < 65536 for c in codes):
            valid_samples.append(f"{filename}|{text}|{json.dumps(codes)}\n")
        else:
            err_encode += 1
            
    except Exception as e:
        if err_encode == 0: print(f"\n❌ Lỗi quá tải/Tensor ({filename}): {e}")
        err_encode += 1
        continue

with open(ENCODED_PATH, 'w', encoding='utf-8') as f:
    f.writelines(valid_samples)

print(f"\n🎉 HOÀN TẤT! Đã mã hóa thành công: {len(valid_samples)} file.")
print(f"📉 BẢNG BÁO CÁO LỖI (Số file bị vứt bỏ):")
print(f"  - Mất do KHÔNG TÌM THẤY file .wav: {err_path} file")
print(f"  - Mất do thời lượng quá ngắn/dài: {err_dur} file")
print(f"  - Mất do không có văn bản: {err_text} file")
print(f"  - Mất do file .wav bị lỗi/hỏng: {err_librosa} file")
print(f"  - Mất do lỗi mô hình NeuCodec: {err_encode} file")

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


🔍 Bắt đầu KIỂM TRA LỖI từ: /kaggle/input/datasets/phmngcduy/data-utterances
✅ Đã tìm thấy 1553 câu nói của SPEAKER_00 trong file CSV. Bắt đầu mã hóa...


config.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:189: UserWarning: The `resume_download` argument is deprecated and ignored in `hf_hub_download`. Downloads always resume whenever possible.
  warnings.warn(


pytorch_model.bin:   0%|          | 0.00/1.16G [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/2.32G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/773 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/275 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)
Đang mã hóa: 100%|██████████| 1553/1553 [06:02<00:00,  4.29it/s]


🎉 HOÀN TẤT! Đã mã hóa thành công: 1553 file.
📉 BẢNG BÁO CÁO LỖI (Số file bị vứt bỏ):
  - Mất do KHÔNG TÌM THẤY file .wav: 0 file
  - Mất do thời lượng quá ngắn/dài: 0 file
  - Mất do không có văn bản: 0 file
  - Mất do file .wav bị lỗi/hỏng: 0 file
  - Mất do lỗi mô hình NeuCodec: 0 file


In [4]:
import os
import json
import torch
from torch.utils.data import Dataset
from peft import LoraConfig, TaskType, get_peft_model
from transformers import (
    AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments, 
    default_data_collator
)
from transformers.trainer_utils import get_last_checkpoint

# ==========================================
# 1. TIỀN XỬ LÝ (Giữ nguyên của bạn - Có Padding 2048)
# ==========================================
def preprocess_sample(sample, tokenizer, max_len=2048):
    speech_gen_start = tokenizer.convert_tokens_to_ids('<|SPEECH_GENERATION_START|>')
    speech_gen_end = tokenizer.convert_tokens_to_ids('<|SPEECH_GENERATION_END|>')
    ignore_index = -100
    
    # 1. Thêm 5 token im lặng vào đuôi để dạy mô hình chủ động "ngắt micro"
    silence_tokens = [0] * 5  
    extended_codes = sample["codes"] + silence_tokens

    chat = (
        f"<|TEXT_PROMPT_START|>{sample['phones']}<|TEXT_PROMPT_END|>"
        f"<|SPEECH_GENERATION_START|>"
        + "".join([f"<|speech_{i}|>" for i in extended_codes])
        + "<|SPEECH_GENERATION_END|>"
    )
    
    raw_ids = tokenizer.encode(chat)
    actual_len = len(raw_ids)
    ids = (raw_ids + [tokenizer.pad_token_id] * max(0, max_len - actual_len))[:max_len]
    
    input_ids = torch.tensor(ids, dtype=torch.long)
    labels = torch.full_like(input_ids, ignore_index)
    
    speech_gen_start_idx = (input_ids == speech_gen_start).nonzero(as_tuple=True)[0]
    speech_gen_end_idx = (input_ids == speech_gen_end).nonzero(as_tuple=True)[0]
    
    # FIX: Chỉ tính Loss từ START đến END, gạt bỏ hoàn toàn vùng PADDING phía sau
    if len(speech_gen_start_idx) > 0 and len(speech_gen_end_idx) > 0:
        start_pos = speech_gen_start_idx[0]
        end_pos = speech_gen_end_idx[0] + 1 
        labels[start_pos:end_pos] = input_ids[start_pos:end_pos]
        
    return {"input_ids": input_ids, "labels": labels, "attention_mask": (input_ids != tokenizer.pad_token_id).long()}

class VieNeuDataset(Dataset):
    def __init__(self, metadata_path, tokenizer):
        self.samples = []
        self.tokenizer = tokenizer
        with open(metadata_path, 'r', encoding='utf-8') as f:
            for line in f:
                parts = line.strip().split('|')
                self.samples.append({"text": parts[1], "codes": json.loads(parts[2])})
    
    def __len__(self): return len(self.samples)
    
    def __getitem__(self, idx):
        sample = self.samples[idx]
        return preprocess_sample({"phones": phonemize_with_dict(sample["text"]), "codes": sample["codes"]}, self.tokenizer)

# ==========================================
# 2. KHỞI TẠO MODEL (Giữ nguyên của bạn - bf16, 2 GPU)
# ==========================================
MODEL_ID = "pnnbao-ump/VieNeu-TTS-0.3B"
print(f"Đang tải mô hình gốc: {MODEL_ID}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.bfloat16, device_map="auto")

lora_config = LoraConfig(
    r=32,                  # Tăng từ 16 lên 32
    lora_alpha=64,         # Tăng từ 32 lên 64 (luôn gấp đôi r)
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj", 
        "gate_proj", "up_proj", "down_proj"     # Học sâu thêm vào tầng MLP để tránh nói ngọng
    ],
    lora_dropout=0.05, 
    bias="none", 
    task_type=TaskType.CAUSAL_LM
)
model = get_peft_model(model, lora_config)

# ==========================================
# 3. DATASET
# ==========================================
full_dataset = VieNeuDataset(ENCODED_PATH, tokenizer)
val_size = max(1, int(0.05 * len(full_dataset)))
train_data, eval_data = torch.utils.data.random_split(full_dataset, [len(full_dataset) - val_size, val_size])

# ==========================================
# 4. TRAINING ARGUMENTS (VÁ LỖI TẠI ĐÂY)
# ==========================================
OUTPUT_DIR = "/kaggle/working/output/VieNeu-LoRA"

args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    max_steps=1000,                      
    per_device_train_batch_size=1,       
    per_device_eval_batch_size=1,  
    eval_accumulation_steps=1,     
    gradient_accumulation_steps=8,       
    gradient_checkpointing=True,         
    
    bf16=True,                           # BẬT BF16 (Đồng bộ mô hình gốc)
    fp16=False,                          # TẮT FP16 (Tránh tràn số)
    
    optim="adamw_8bit",                  
    learning_rate=1e-4,                  # Giảm từ 2e-4 xuống 1e-4 để tránh nhảy quá đà
    weight_decay=0.01,                   # Thêm kiểm soát Overfitting trực tiếp
    warmup_ratio=0.1,                    
    logging_steps=20,
    report_to="none",
    
    save_steps=100,              
    eval_strategy="steps",
    eval_steps=100,              
    save_total_limit=3,                  
    load_best_model_at_end=True,         # Tự động nạp bản tốt nhất khi kết thúc
    metric_for_best_model="loss"         # Tiêu chí chọn: Val Loss thấp nhất
)

trainer = Trainer(
    model=model, 
    args=args, 
    train_dataset=train_data, 
    eval_dataset=eval_data, 
    data_collator=default_data_collator
)

# ==========================================
# 5. TỰ ĐỘNG TÌM KÝ ỨC & TRAIN
# ==========================================
last_checkpoint = None
if os.path.isdir(OUTPUT_DIR):
    last_checkpoint = get_last_checkpoint(OUTPUT_DIR)

if last_checkpoint is not None:
    print(f"\n🔄 TÌM THẤY CHECKPOINT! Đang tiếp tục Train từ mốc: {last_checkpoint}...")
    trainer.train(resume_from_checkpoint=last_checkpoint)
else:
    print("\n🚀 Không có Checkpoint. Bắt đầu Train từ con số 0...")
    trainer.train()

# Lưu Model cuối
model.save_pretrained(f"{OUTPUT_DIR}-Final")
tokenizer.save_pretrained(f"{OUTPUT_DIR}-Final")
print(f"🎉 Model LoRA đã được lưu tại: {OUTPUT_DIR}-Final")

Đang tải mô hình gốc: pnnbao-ump/VieNeu-TTS-0.3B


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/12.1M [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt:   0%|          | 0.00/758 [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/169 [00:00<?, ?B/s]

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.



🚀 Không có Checkpoint. Bắt đầu Train từ con số 0...


Step,Training Loss,Validation Loss
100,7.242998,7.402644
200,6.966586,7.220708
300,6.825217,7.175243
400,6.752657,7.160965
500,6.657916,7.174258
600,6.561424,7.195760
700,6.515517,7.225911
800,6.425179,7.255569
900,6.407540,7.285900
1000,6.346710,7.304276


🎉 Model LoRA đã được lưu tại: /kaggle/working/output/VieNeu-LoRA-Final
